# Module 05: Neural Networks from Scratch
## Chapter 5.5 — Backpropagation from Scratch: The Crown Jewel of Deep Learning

> **"If forward propagation is how a neural network speaks, backpropagation is how it listens, learns, and corrects its mistakes."**

---

### Executive Overview & Learning Path

In **1986**, David Rumelhart, Geoffrey Hinton, and Ronald Williams published a paper in *Nature* titled *"Learning Internal Representations by Error Propagation"*. It popularized an algorithm that transformed artificial neural networks from an academic curiosity into the dominant computing paradigm on Earth: **Backpropagation**.

At its core, backpropagation is not mysterious magic; it is simply the **multivariate chain rule of calculus** implemented efficiently via dynamic programming on a directed acyclic computational graph. 

In this chapter, we derive backpropagation mathematically from the ground up, translate every step into clean batched NumPy matrix equations, implement a complete training loop from scratch, and rigorously prove our calculus using **Numerical Gradient Checking** ($\epsilon$-verification).

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                               CHAPTER 5.5 ROADMAP                                      │
├────────────────────────────────────────────────────────────────────────────────────────┤
│ 1. THE COMPUTATIONAL GRAPH   ──> Dynamic Programming, Node Gates, & The Chain Rule     │
│ 2. STEP-BY-STEP MATRIX DERIV ──> Analytical Gradients: dL/dW1, dL/db1, dL/dW2, dL/db2   │
│ 3. MANUAL BACKPROP ENGINE    ──> Pure NumPy 2-Layer MLP Forward & Backward Implementation│
│ 4. NUMERICAL GRADIENT CHECK  ──> Two-Sided Finite Difference Verification (Error < 1e-7)│
│ 5. END-TO-END TRAINING LOOP  ──> Training on Non-Linear Moons Data & Boundary Plotting │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

---


---

## 1. The Computational Graph — Reverse Dynamic Programming & The Chain Rule

---

### 🗣️ In Simple English

How can a single scalar error (e.g., *"Your loss is 1.84"*) tell **100 million distinct weights** in a deep network exactly how much each individual parameter contributed to the mistake?

Think of it this way: Imagine a large automobile assembly line with 50 workers:
- Worker 1 installs the engine bolts.
- Worker 10 mounts the wheels.
- Worker 40 connects the brake lines.
- At the end of the line, a car fails its brake inspection by $3$ meters.

To fix the car, you don't disassemble the entire vehicle randomly. You trace the defect backwards down the assembly line:
1. You inspect the brake fluid pressure at the calipers (Worker 50).
2. You ask: *"How much did the master cylinder pressure affect the caliper pressure?"* (Local derivative of Worker 40).
3. By multiplying the downstream defect signal by each worker's personal contribution, you calculate the exact blame for every wrench turn in the entire factory!

The key point is: This reverse blame assignment is **The Chain Rule**. In forward propagation, data flows forward through arithmetic gates. In backward propagation, each gate takes the incoming blame from the right, multiplies it by its own local derivative, and passes the blame to the left!

---

### 🌍 Real-Life Example

**Scenario:** Calculating profit sensitivity in a multi-stage corporate supply chain.

Let total profit $P$ depend on retail sales revenue $R$, which depends on units shipped $U$, which depends on factory raw steel supply $S$:
$$S \xrightarrow{\text{Factory}} U(S) \xrightarrow{\text{Distributor}} R(U) \xrightarrow{\text{Retail}} P(R)$$

| Calculus Term | Supply Chain Meaning | Numerical Intuition |
|:---|:---|:---|
| $\frac{dP}{dR} = 0.20$ | Profit margin per dollar of retail revenue | Every \$1.00 in revenue yields \$0.20 in profit |
| $\frac{dR}{dU} = 50.0$ | Revenue generated per unit shipped | Each car sold brings in \$50.00 |
| $\frac{dU}{dS} = 2.0$ | Cars manufactured per ton of raw steel | 1 ton of steel produces 2 cars |
| $\frac{dP}{dS} = \frac{dP}{dR} \cdot \frac{dR}{dU} \cdot \frac{dU}{dS}$ | **The Chain Rule:** Profit change per ton of steel | $0.20 \times 50.0 \times 2.0 = \mathbf{\$20.00}$ per ton of steel! |

**Why this matters:** You do not need to rewrite the entire company profit equation as an immense polynomial of steel tonnage. You simply multiply the local transmission rates along the chain.

---

### 📖 How It Works (Step by Step)

#### A. The Single-Variable Chain Rule
If variable $z$ depends on $y$, and $y$ depends on $x$:
$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$$

#### B. The Multi-Variable (Branching) Chain Rule
If variable $x$ feeds into multiple intermediate nodes $y_1, y_2, \dots, y_k$, and all intermediate nodes influence final scalar loss $L$:
$$\frac{\partial L}{\partial x} = \sum_{j=1}^k \frac{\partial L}{\partial y_j} \cdot \frac{\partial y_j}{\partial x}$$
*Gradients from multiple downstream paths simply add together!*

#### C. The Anatomy of an Arithmetic Node Gate
Every operation in a neural network (addition, multiplication, matrix product, activation) can be viewed as a node in a directed acyclic graph:
1. **Forward Pass:** Receives inputs $x$ and $y$. Computes output $z = f(x, y)$ and **caches inputs in memory**.
2. **Backward Pass:** Receives incoming upstream gradient $\frac{\partial L}{\partial z}$ from downstream layers.
3. Computes local derivatives: $\frac{\partial z}{\partial x}$ and $\frac{\partial z}{\partial y}$.
4. Computes outgoing downstream gradients via multiplication:
   $$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial x}, \quad \frac{\partial L}{\partial y} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial y}$$

---

### 🔑 Key Concepts

| Concept | Plain English Definition | Why It Matters |
|:---|:---|:---|
| **Forward Pass** | Computing intermediate layer activations from inputs to outputs. | Tensors must be stored (cached) in memory for use during the backward pass. |
| **Backward Pass** | Propagating gradients from loss back to parameters. | Reuses cached forward activations to calculate parameter updates. |
| **Memory Bottleneck (Activation Caching)** | The reason training requires far more GPU VRAM than inference. | Every hidden activation $A^{[l]}$ must be held in RAM until backpropagation reaches it. |

---

### 🗺️ Visual Overview

```
THE ANATOMY OF A COMPUTATIONAL GRAPH GATE:

        FORWARD PASS                                 BACKWARD PASS
   (Data flows left to right)                  (Gradients flow right to left)

x ────\                                   dL/dx = (dL/dz) * (dz/dx) <────\
       ───► [ Gate: z = f(x, y) ] ───► z                                  ─── [ Local Gate ] <─── dL/dz (Upstream)
y ────/                                   dL/dy = (dL/dz) * (dz/dy) <────/

Example: Multiplication Gate z = x * y
  Forward:  z = x * y
  Backward: dL/dx = (dL/dz) * y       (The multiplier simply swaps the other input!)
            dL/dy = (dL/dz) * x
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Backpropagation is an optimization algorithm that updates weights."* | **False!** Backpropagation **only calculates gradients** ($\frac{\partial L}{\partial W}$). It does not change weights by a single micro-fraction. Updating the weights using those gradients is the separate job of the **Optimizer** (e.g., SGD, Adam). |
| *"You need to calculate symbolic algebraic equations for the entire deep network."* | **False.** We never derive a gigantic closed-form formula for the entire network. We only ever write the tiny local derivative for each single gate! Dynamic programming chains them together automatically. |

---

### ✅ When to Use This

- Computational graphs form the exact autograd engine powering **PyTorch (`torch.autograd`)** and **TensorFlow (`tf.GradientTape`)**.

---

### 📊 Section Quick Reference

| Elementary Gate | Forward Formula ($z$) | Local Derivative $\frac{\partial z}{\partial x}$ | Outgoing Gradient $\frac{\partial L}{\partial x}$ |
|:---|:---:|:---:|:---:|
| **Addition Gate** | $x + y$ | $1$ | $\frac{\partial L}{\partial z}$ (Gradient Distributor) |
| **Multiplication Gate** | $x \cdot y$ | $y$ | $\frac{\partial L}{\partial z} \cdot y$ (Gradient Switcher) |
| **ReLU Gate** | $\max(0, x)$ | $1 \text{ if } x > 0 \text{ else } 0$ | $\frac{\partial L}{\partial z} \cdot \mathbf{1}_{\{x > 0\}}$ (Gradient Gate) |

---


In [1]:
# ✅ Code Demo: Elemental Computational Graph Node with Forward & Backward Autograd
# Concept: Forward Evaluation, Intermediate Caching, and Reverse Chain Rule Multiplication
# -----------------------------------------------------------------------------------------

import os
import numpy as np

# 1. Scalar Computational Graph Arithmetic Function: f(x, y, z) = (x + y) * z
# Let x = -2, y = 5, z = -4
x = -2.0
y = 5.0
z = -4.0

# Forward Pass (with intermediate caching)
q = x + y       # Addition gate: q = 3.0
f = q * z       # Multiplication gate: f = -12.0

print("=" * 65)
print("COMPUTATIONAL GRAPH FORWARD & BACKWARD TRACE")
print("=" * 65)
print(f"Forward Pass: f(x, y, z) = (x + y) * z")
print(f"  Inputs: x = {x}, y = {y}, z = {z}")
print(f"  Intermediate Gate q = x + y = {q}")
print(f"  Final Output f = q * z      = {f}")
print("-" * 65)

# Backward Pass (Chain Rule Step-by-Step)
# Seed gradient: df/df = 1.0
df_df = 1.0

# Gate 2: f = q * z
# df/dz = df/df * (df/dz) = 1.0 * q
df_dz = df_df * q
# df/dq = df/df * (df/dq) = 1.0 * z
df_dq = df_df * z

# Gate 1: q = x + y
# df/dx = df/dq * (dq/dx) = df/dq * 1.0
df_dx = df_dq * 1.0
# df/dy = df/dq * (dq/dy) = df/dq * 1.0
df_dy = df_dq * 1.0

print("Backward Pass (Analytical Gradients via Chain Rule):")
print(f"  df/dz = q     = {df_dz:+.4f} (If z increases by 1, f increases by {df_dz})")
print(f"  df/dq = z     = {df_dq:+.4f}")
print(f"  df/dx = df/dq = {df_dx:+.4f} (If x increases by 1, f changes by {df_dx})")
print(f"  df/dy = df/dq = {df_dy:+.4f} (If y increases by 1, f changes by {df_dy})")
print("-" * 65)

# 2. Numerical Verification via Finite Differences: f(x + eps) - f(x) / eps
eps = 1e-6
num_df_dx = (((x + eps) + y) * z - f) / eps
num_df_dy = ((x + (y + eps)) * z - f) / eps
num_df_dz = ((x + y) * (z + eps) - f) / eps

print("Numerical Finite Difference Verification:")
print(f"  Analytical df/dx: {df_dx:+.6f} | Numerical: {num_df_dx:+.6f} | Match: {np.isclose(df_dx, num_df_dx)}")
print(f"  Analytical df/dy: {df_dy:+.6f} | Numerical: {num_df_dy:+.6f} | Match: {np.isclose(df_dy, num_df_dy)}")
print(f"  Analytical df/dz: {df_dz:+.6f} | Numerical: {num_df_dz:+.6f} | Match: {np.isclose(df_dz, num_df_dz)}")
print("=" * 65)


COMPUTATIONAL GRAPH FORWARD & BACKWARD TRACE
Forward Pass: f(x, y, z) = (x + y) * z
  Inputs: x = -2.0, y = 5.0, z = -4.0
  Intermediate Gate q = x + y = 3.0
  Final Output f = q * z      = -12.0
-----------------------------------------------------------------
Backward Pass (Analytical Gradients via Chain Rule):
  df/dz = q     = +3.0000 (If z increases by 1, f increases by 3.0)
  df/dq = z     = -4.0000
  df/dx = df/dq = -4.0000 (If x increases by 1, f changes by -4.0)
  df/dy = df/dq = -4.0000 (If y increases by 1, f changes by -4.0)
-----------------------------------------------------------------
Numerical Finite Difference Verification:
  Analytical df/dx: -4.000000 | Numerical: -4.000000 | Match: True
  Analytical df/dy: -4.000000 | Numerical: -4.000000 | Match: True
  Analytical df/dz: +3.000000 | Numerical: +3.000000 | Match: True


---

## 2. Matrix Calculus & Neural Network Gradients Step-by-Step

---

### 🗣️ In Simple English

When we move from simple scalars ($x, y, z$) to batches of matrices containing millions of numbers, how do we take derivatives?

The secret is **Dimension Consistency Matching**.
- If a weight matrix $W^{[1]}$ has shape $(d, h)$ (say $4 \times 8$), then its gradient matrix $\frac{\partial J}{\partial W^{[1]}}$ **MUST have the exact same shape $(4 \times 8)$**!
- Why? Because every single weight $W_{ij}$ needs its own personal gradient $\frac{\partial J}{\partial W_{ij}}$ so the optimizer can update it:
  $$W_{ij} \leftarrow W_{ij} - \eta \frac{\partial J}{\partial W_{ij}}$$

By tracking the matrix dimensions and transposing matrices appropriately, the chain rule formulas for an entire layer collapse into a few lines of matrix algebra!

---

### 🌍 Real-Life Example

**Scenario:** Training a 2-layer neural network on a batch of $N = 64$ patients with $d = 4$ biomarkers, $h = 8$ hidden neurons, and $c = 3$ medical diagnosis classes.

| Matrix Variable | Shape | What It Represents |
|:---:|:---:|:---|
| $X$ | $(64, 4)$ | Input batch of 64 patients, 4 biomarkers each |
| $W^{[1]}$ | $(4, 8)$ | Synaptic connection weights from 4 biomarkers to 8 hidden risk factors |
| $\mathbf{b}^{[1]}$ | $(1, 8)$ | Baseline activation thresholds for the 8 hidden risk factors |
| $Z^{[1]}$ | $(64, 8)$ | Hidden pre-activations: $Z^{[1]} = X W^{[1]} + \mathbf{b}^{[1]}$ |
| $A^{[1]}$ | $(64, 8)$ | Hidden non-linear activations: $A^{[1]} = \text{ReLU}(Z^{[1]})$ |
| $W^{[2]}$ | $(8, 3)$ | Weights from 8 hidden factors to 3 diagnosis classes |
| $\mathbf{b}^{[2]}$ | $(1, 3)$ | Output layer biases |
| $Z^{[2]}$ | $(64, 3)$ | Output logits |
| $\hat{Y}$ | $(64, 3)$ | Calibrated Softmax diagnosis probabilities |
| $Y$ | $(64, 3)$ | True diagnosis labels (One-hot encoded) |

---

### 📖 How It Works (Step by Step)

Let $J = \text{CCE}(Y, \hat{Y})$ be Categorical Cross-Entropy loss.

#### Step 1: Output Layer Gradient ($\delta^{[2]}$)
As proven in **Chapter 5.4**, the derivative of Softmax + Cross-Entropy with respect to output logits $Z^{[2]}$ is the prediction error:
$$\delta^{[2]} = \frac{\partial J}{\partial Z^{[2]}} = \frac{1}{N} (\hat{Y} - Y) \quad \in \mathbb{R}^{N \times c}$$

#### Step 2: Gradients for Layer 2 Parameters ($W^{[2]}$ and $\mathbf{b}^{[2]}$)
Since $Z^{[2]} = A^{[1]} W^{[2]} + \mathbf{b}^{[2]}$:
- **Gradient w.r.t $W^{[2]}$:**
  $$\frac{\partial J}{\partial W^{[2]}} = (A^{[1]})^T \delta^{[2]}$$
  - *Shape Check:* $(h, N) \times (N, c) = (h, c)$ ✅ Matches $W^{[2]}$!
- **Gradient w.r.t $\mathbf{b}^{[2]}$:**
  $$\frac{\partial J}{\partial \mathbf{b}^{[2]}} = \sum_{i=1}^N \delta_i^{[2]} = \text{np.sum}(\delta^{[2]}, \text{axis}=0, \text{keepdims}=\text{True})$$
  - *Shape Check:* Summing down all $N$ batch rows produces $(1, c)$ ✅ Matches $\mathbf{b}^{[2]}$!

#### Step 3: Backpropagating Error through the Hidden Layer ($\delta^{[1]}$)
Now we pull the error gradient backwards through the weights $W^{[2]}$ into the hidden layer activations $A^{[1]}$:
$$\frac{\partial J}{\partial A^{[1]}} = \delta^{[2]} (W^{[2]})^T$$
- *Shape Check:* $(N, c) \times (c, h) = (N, h)$ ✅

Next, propagate through the ReLU activation function using the element-wise Hadamard product ($\odot$):
$$\delta^{[1]} = \frac{\partial J}{\partial Z^{[1]}} = \frac{\partial J}{\partial A^{[1]}} \odot \text{ReLU}'(Z^{[1]})$$
where $\text{ReLU}'(Z^{[1]}) = 1$ if $Z^{[1]} > 0$ else $0$.

#### Step 4: Gradients for Layer 1 Parameters ($W^{[1]}$ and $\mathbf{b}^{[1]}$)
Since $Z^{[1]} = X W^{[1]} + \mathbf{b}^{[1]}$:
- **Gradient w.r.t $W^{[1]}$:**
  $$\frac{\partial J}{\partial W^{[1]}} = X^T \delta^{[1]}$$
  - *Shape Check:* $(d, N) \times (N, h) = (d, h)$ ✅ Matches $W^{[1]}$!
- **Gradient w.r.t $\mathbf{b}^{[1]}$:**
  $$\frac{\partial J}{\partial \mathbf{b}^{[1]}} = \sum_{i=1}^N \delta_i^{[1]} = \text{np.sum}(\delta^{[1]}, \text{axis}=0, \text{keepdims}=\text{True})$$
  - *Shape Check:* Summing down batch rows produces $(1, h)$ ✅ Matches $\mathbf{b}^{[1]}$!

---

### 🔑 Key Concepts

| Step | Operation | Math Formula | NumPy Implementation |
|:---|:---|:---|:---|
| **Output Error** | Logit Gradient | $\delta^{[2]} = \frac{1}{N}(\hat{Y} - Y)$ | `dZ2 = (Y_hat - Y) / N` |
| **Layer 2 Weights** | Dot Product | $(A^{[1]})^T \delta^{[2]}$ | `dW2 = np.dot(A1.T, dZ2)` |
| **Layer 2 Biases** | Row Summation | $\sum \delta^{[2]}$ | `db2 = np.sum(dZ2, axis=0, keepdims=True)` |
| **Hidden Error** | Linear Backprop | $\delta^{[2]} (W^{[2]})^T \odot \sigma'$ | `dZ1 = np.dot(dZ2, W2.T) * (Z1 > 0)` |
| **Layer 1 Weights** | Dot Product | $X^T \delta^{[1]}$ | `dW1 = np.dot(X.T, dZ1)` |
| **Layer 1 Biases** | Row Summation | $\sum \delta^{[1]}$ | `db1 = np.sum(dZ1, axis=0, keepdims=True)` |

---

### 🗺️ Visual Overview

```
THE FULL MATRIX BACKPROPAGATION SIGNAL FLOW:

  FORWARD:
  X (N, d) ──> [ @ W1 + b1 ] ──> Z1 (N, h) ──> [ ReLU ] ──> A1 (N, h) ──> [ @ W2 + b2 ] ──> Z2 (N, c) ──> [ Softmax ] ──> Y_hat
                                                                                                                   │
                                                                                                                   ▼
  BACKWARD:                                                                                                     Loss J
  dW1 = X.T @ dZ1 <─── dZ1 = dA1 * (Z1>0) <─── dA1 = dZ2 @ W2.T <─── dW2 = A1.T @ dZ2 <─── dZ2 = (Y_hat - Y) / N <───┘
  db1 = sum(dZ1)                                                      db2 = sum(dZ2)
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"In `dW1 = np.dot(X.T, dZ1)`, why is $X$ transposed? Shouldn't it be $X$ times $dZ1$?"* | **Matrix dimension rule!** $X$ has shape $(N, d)$ and $\delta^{[1]}$ has shape $(N, h)$. You cannot multiply $(N, d) \times (N, h)$ because the inner dimensions don't match ($d \ne N$). Transposing $X$ to $(d, N)$ produces $(d, N) \times (N, h) = (d, h)$, which perfectly matches $W^{[1]}$ and sums the gradient contributions across all $N$ batch samples! |

---

### ✅ When to Use This

- This is the exact backward engine executed billions of times daily in modern deep learning training pipelines.

---

### 📊 Section Quick Reference

| Layer Parameter | Parameter Shape | Gradient Formula | Gradient Shape |
|:---:|:---:|:---|:---:|
| $W^{[2]}$ | $(h, c)$ | $(A^{[1]})^T \delta^{[2]}$ | $(h, c)$ ✅ |
| $\mathbf{b}^{[2]}$ | $(1, c)$ | $\sum \delta^{[2]}$ | $(1, c)$ ✅ |
| $W^{[1]}$ | $(d, h)$ | $X^T \delta^{[1]}$ | $(d, h)$ ✅ |
| $\mathbf{b}^{[1]}$ | $(1, h)$ | $\sum \delta^{[1]}$ | $(1, h)$ ✅ |

---


In [2]:
# ✅ Code Demo: Pure NumPy 2-Layer MLP Forward and Analytical Backward Pass Engine
# Concept: Forward Evaluation, Intermediate Caching, and Matrix Gradient Derivations
# ----------------------------------------------------------------------------------

import os
import numpy as np

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

class TwoLayerNeuralNetwork:
    def __init__(self, in_features, hidden_dim, out_classes, seed=42):
        np.random.seed(seed)
        # He Normal initialization for layer 1 (ReLU)
        self.W1 = np.random.randn(in_features, hidden_dim) * np.sqrt(2.0 / in_features)
        self.b1 = np.zeros((1, hidden_dim))
        
        # Xavier Normal initialization for layer 2 (Softmax)
        self.W2 = np.random.randn(hidden_dim, out_classes) * np.sqrt(1.0 / hidden_dim)
        self.b2 = np.zeros((1, out_classes))
        
        # Cache for backward pass
        self.cache = {}

    def forward(self, X):
        # Layer 1
        Z1 = np.dot(X, self.W1) + self.b1
        A1 = np.maximum(0, Z1)  # ReLU
        
        # Layer 2
        Z2 = np.dot(A1, self.W2) + self.b2
        
        # Numerically stable Softmax
        max_Z2 = np.max(Z2, axis=1, keepdims=True)
        exp_Z2 = np.exp(Z2 - max_Z2)
        Y_hat = exp_Z2 / np.sum(exp_Z2, axis=1, keepdims=True)
        
        # Cache tensors for backward pass
        self.cache = {'X': X, 'Z1': Z1, 'A1': A1, 'Z2': Z2, 'Y_hat': Y_hat}
        return Y_hat

    def compute_loss(self, Y_hat, Y_true):
        N = Y_true.shape[0]
        # Cross-Entropy Loss (with eps clipping)
        eps = 1e-15
        loss = -np.sum(Y_true * np.log(np.clip(Y_hat, eps, 1.0))) / N
        return loss

    def backward(self, Y_true):
        # Retrieve cached forward tensors
        X = self.cache['X']
        Z1 = self.cache['Z1']
        A1 = self.cache['A1']
        Y_hat = self.cache['Y_hat']
        N = X.shape[0]

        # Step 1: Output layer gradient (Softmax + CCE)
        dZ2 = (Y_hat - Y_true) / N
        
        # Step 2: Layer 2 parameter gradients
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        
        # Step 3: Backprop into hidden layer
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (Z1 > 0)  # Derivative of ReLU
        
        # Step 4: Layer 1 parameter gradients
        dW1 = np.dot(X.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
        return grads

# Test Forward and Backward on a Synthetic Mini-Batch
np.random.seed(42)
N_batch = 5
d_in = 4
h_dim = 6
c_out = 3

X_sample = np.random.randn(N_batch, d_in)
# One-hot target labels
y_indices = np.random.randint(0, c_out, size=N_batch)
Y_sample = np.eye(c_out)[y_indices]

# Instantiate model
model = TwoLayerNeuralNetwork(in_features=d_in, hidden_dim=h_dim, out_classes=c_out)

# Run Forward Pass
Y_preds = model.forward(X_sample)
loss_val = model.compute_loss(Y_preds, Y_sample)

# Run Backward Pass
gradients = model.backward(Y_sample)

print("=" * 65)
print("ANALYTICAL BACKPROPAGATION TENSOR & GRADIENT VERIFICATION")
print("=" * 65)
print(f"Batch Size N = {N_batch}, Input Dim = {d_in}, Hidden = {h_dim}, Classes = {c_out}")
print(f"Computed Cross-Entropy Loss: {loss_val:.4f}\n")
print(f"1. Weight W1 Shape:    {model.W1.shape}  | Gradient dW1 Shape:    {gradients['dW1'].shape} ✅")
print(f"2. Bias b1 Shape:      {model.b1.shape}  | Gradient db1 Shape:    {gradients['db1'].shape} ✅")
print(f"3. Weight W2 Shape:    {model.W2.shape}  | Gradient dW2 Shape:    {gradients['dW2'].shape} ✅")
print(f"4. Bias b2 Shape:      {model.b2.shape}  | Gradient db2 Shape:    {gradients['db2'].shape} ✅")
print("-" * 65)
print(f"Gradient Norms: ||dW1|| = {np.linalg.norm(gradients['dW1']):.6f}, ||dW2|| = {np.linalg.norm(gradients['dW2']):.6f}")
print("=" * 65)


ANALYTICAL BACKPROPAGATION TENSOR & GRADIENT VERIFICATION
Batch Size N = 5, Input Dim = 4, Hidden = 6, Classes = 3
Computed Cross-Entropy Loss: 1.1188

1. Weight W1 Shape:    (4, 6)  | Gradient dW1 Shape:    (4, 6) ✅
2. Bias b1 Shape:      (1, 6)  | Gradient db1 Shape:    (1, 6) ✅
3. Weight W2 Shape:    (6, 3)  | Gradient dW2 Shape:    (6, 3) ✅
4. Bias b2 Shape:      (1, 3)  | Gradient db2 Shape:    (1, 3) ✅
-----------------------------------------------------------------
Gradient Norms: ||dW1|| = 0.468398, ||dW2|| = 1.138008


---

## 3. Numerical Gradient Checking — Mathematical Proof of Zero Bugs

---

### 🗣️ In Simple English

How do engineers at DeepMind, OpenAI, or Meta know for $100\%$ certainty that their backpropagation code doesn't have a hidden sign error or transposed matrix?

They use **Numerical Gradient Checking** ($\epsilon$-verification).

Think of it this way: Imagine you are calibrating a high-tech laser speedometer.
- **Analytical Gradient (Backpropagation):** The theoretical speed calculated instantly using the laws of physics ($v = at$).
- **Numerical Gradient (Finite Difference):** You place two physical cameras exactly $1$ centimeter apart ($\epsilon$) and measure the time it takes the car to cross between them:
  $$\text{Speed} \approx \frac{\text{Distance}}{\text{Time}} = \frac{f(\theta + \epsilon) - f(\theta - \epsilon)}{2\epsilon}$$

If the laser speedometer and the two-camera measurement match to **8 decimal places**, you have mathematical proof that your speedometer works with zero bugs!

---

### 📖 How It Works (Step by Step)

1. **The Two-Sided Difference Formula:**
   For any single scalar parameter $\theta_i$ in the network:
   $$g_{\text{numerical}}(\theta_i) = \frac{J(\theta_1, \dots, \theta_i + \epsilon, \dots) - J(\theta_1, \dots, \theta_i - \epsilon, \dots)}{2\epsilon}$$
   We choose $\epsilon = 10^{-7}$ (in standard 64-bit float).
   *Why two-sided?* A Taylor expansion proves that two-sided difference has an error of $O(\epsilon^2) \approx 10^{-14}$, whereas one-sided difference has an error of $O(\epsilon) \approx 10^{-7}$.
2. **The Vectorized Comparison:**
   Stack all analytical gradients into a single flat vector $\mathbf{g}_{\text{analytical}}$ and all numerical approximations into a single flat vector $\mathbf{g}_{\text{numerical}}$.
3. **The Relative Error Metric:**
   Compute the Euclidean relative difference:
   $$\text{Relative Error} = \frac{\|\mathbf{g}_{\text{analytical}} - \mathbf{g}_{\text{numerical}}\|_2}{\|\mathbf{g}_{\text{analytical}}\|_2 + \|\mathbf{g}_{\text{numerical}}\|_2}$$
4. **The Verdict Decision Rule:**
   - **Relative Error $< 10^{-7}$:** 🎉 **Perfection!** Backpropagation implementation is mathematically verified with zero bugs.
   - **Relative Error $10^{-5}$ to $10^{-7}$:** Acceptable if using non-smooth activations (like ReLU kinks).
   - **Relative Error $> 10^{-4}$:** 🚨 **CRITICAL BUG!** There is an error in your backward calculus!

---

### 🔑 Key Concepts

| Concept | What It Means | Why It Matters |
|:---|:---|:---|
| **Gradient Checking** | Comparing analytical backprop gradients against finite differences. | The gold-standard debugging tool for custom deep learning implementations. |
| **Two-Sided Difference** | $[J(\theta+\epsilon) - J(\theta-\epsilon)] / 2\epsilon$. | Far more accurate than one-sided difference $[J(\theta+\epsilon) - J(\theta)]/\epsilon$. |
| **Training Infeasibility** | Gradient checking requires $2 \times P$ forward passes (where $P$ is parameter count). | Far too slow for actual training; only run during debugging on tiny batches. |

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Can we use gradient checking during regular model training?"* | **Impossible!** For a small model with 100,000 parameters, one step of gradient checking requires **200,000 forward passes**! In contrast, backpropagation computes all 100,000 gradients in **a single backward pass**. Backprop is roughly $100,000\times$ faster! |

---

### ✅ When to Use This

- Always run gradient checking on custom layers or loss functions before launching large-scale training jobs.

---


In [3]:
# ✅ Code Demo: Numerical Gradient Checking (Finite Differences vs Backprop)
# Concept: Two-Sided Epsilon Difference, Parameter Vectorization, and Relative Error
# -----------------------------------------------------------------------------------

import numpy as np

def gradient_check(model, X, Y, eps=1e-7):
    # 1. Compute analytical gradients via backpropagation
    Y_pred = model.forward(X)
    analytical_grads = model.backward(Y)
    
    # Pack analytical gradients into a single flat vector
    grad_analytical_vec = np.concatenate([
        analytical_grads['dW1'].ravel(),
        analytical_grads['db1'].ravel(),
        analytical_grads['dW2'].ravel(),
        analytical_grads['db2'].ravel()
    ])
    
    # 2. Compute numerical gradients via two-sided finite difference
    param_list = [model.W1, model.b1, model.W2, model.b2]
    numerical_grads = []
    
    print("=" * 65)
    print("NUMERICAL GRADIENT CHECKING VERIFICATION (eps = 1e-7)")
    print("=" * 65)
    
    for p_idx, param in enumerate(param_list):
        p_name = ['W1', 'b1', 'W2', 'b2'][p_idx]
        flat_param = param.ravel()
        
        for i in range(len(flat_param)):
            orig_val = flat_param[i]
            
            # f(theta + eps)
            flat_param[i] = orig_val + eps
            loss_plus = model.compute_loss(model.forward(X), Y)
            
            # f(theta - eps)
            flat_param[i] = orig_val - eps
            loss_minus = model.compute_loss(model.forward(X), Y)
            
            # Reset parameter
            flat_param[i] = orig_val
            
            # Two-sided numerical gradient
            num_grad = (loss_plus - loss_minus) / (2.0 * eps)
            numerical_grads.append(num_grad)
            
    grad_numerical_vec = np.array(numerical_grads)
    
    # 3. Compute Euclidean Relative Error
    numerator = np.linalg.norm(grad_analytical_vec - grad_numerical_vec)
    denominator = np.linalg.norm(grad_analytical_vec) + np.linalg.norm(grad_numerical_vec)
    relative_error = numerator / (denominator + 1e-15)
    
    print(f"Total Parameters Checked: {len(grad_analytical_vec)}")
    print(f"Analytical Vector Norm:   {np.linalg.norm(grad_analytical_vec):.8f}")
    print(f"Numerical Vector Norm:    {np.linalg.norm(grad_numerical_vec):.8f}")
    print(f"Difference Norm:          {numerator:.8e}")
    print(f"Relative Error:           {relative_error:.8e}")
    print("-" * 65)
    
    if relative_error < 1e-7:
        print("VERDICT: GRADIENT CHECK PASSED PERFECTLY! (Error < 1e-7)")
        print("Mathematical proof that manual backpropagation is 100% bug-free!")
    elif relative_error < 1e-5:
        print("VERDICT: ACCEPTABLE (Minor floating-point variance due to ReLU kink).")
    else:
        print("VERDICT: FAILED! Check analytical backward derivations for bugs.")
    print("=" * 65)
    return relative_error

# Run gradient check on a small precision test batch
np.random.seed(101)
test_model = TwoLayerNeuralNetwork(in_features=3, hidden_dim=4, out_classes=2)
X_test = np.random.randn(4, 3)
Y_test = np.array([[1, 0], [0, 1], [1, 0], [0, 1]])

rel_err = gradient_check(test_model, X_test, Y_test, eps=1e-7)


NUMERICAL GRADIENT CHECKING VERIFICATION (eps = 1e-7)
Total Parameters Checked: 26
Analytical Vector Norm:   0.40349592
Numerical Vector Norm:    0.40349592
Difference Norm:          1.97432659e-09
Relative Error:           2.44652606e-09
-----------------------------------------------------------------
VERDICT: GRADIENT CHECK PASSED PERFECTLY! (Error < 1e-7)
Mathematical proof that manual backpropagation is 100% bug-free!


---

## 4. Complete Training Loop — Learning the Non-Linear Moons Dataset

---

### 🗣️ In Simple English

Now that our manual backpropagation engine has passed gradient checking with mathematical perfection, let's put it to work!

We will train our pure NumPy neural network on the classic **Non-Linear Half-Moons Dataset** — two interlocking crescent moons that are completely impossible for any linear perceptron to classify.

The training loop follows five rhythmic heartbeats repeated every epoch:
1. **Forward Pass:** Compute predictions $\hat{Y} = \text{forward}(X)$.
2. **Loss Computation:** Evaluate cost $J = \text{compute\_loss}(\hat{Y}, Y)$.
3. **Backward Pass:** Compute analytical gradients $\text{backward}(Y)$.
4. **Parameter Update:** Nudge weights downhill: $W \leftarrow W - \eta \cdot \nabla W$.
5. **Zero Gradients / Track Progress:** Record loss and accuracy to monitor convergence!

---


In [4]:
# ✅ Code Demo: Complete NumPy Neural Network Training Loop on Non-Linear Moons
# Concept: Iterative Gradient Updates, Loss Convergence, and Non-Linear Decision Surface
# -------------------------------------------------------------------------------------

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-05-Neural-Networks/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Generate Synthetic Two-Moons Non-Linear Dataset in Pure NumPy
def make_moons_numpy(n_samples=300, noise=0.15, seed=42):
    np.random.seed(seed)
    n_samples_out = n_samples // 2
    n_samples_in = n_samples - n_samples_out
    
    # Outer moon
    outer_circ_x = np.cos(np.linspace(0, np.pi, n_samples_out))
    outer_circ_y = np.sin(np.linspace(0, np.pi, n_samples_out))
    
    # Inner moon
    inner_circ_x = 1 - np.cos(np.linspace(0, np.pi, n_samples_in))
    inner_circ_y = 1 - np.sin(np.linspace(0, np.pi, n_samples_in)) - 0.5
    
    X = np.vstack([
        np.column_stack([outer_circ_x, outer_circ_y]),
        np.column_stack([inner_circ_x, inner_circ_y])
    ])
    y = np.hstack([np.zeros(n_samples_out, dtype=int), np.ones(n_samples_in, dtype=int)])
    
    # Add Gaussian noise
    X += np.random.randn(*X.shape) * noise
    return X, y

X_moons, y_moons = make_moons_numpy(n_samples=400, noise=0.15)
Y_moons_onehot = np.eye(2)[y_moons]

# 2. Initialize Model: Input(2) -> Hidden(16, ReLU) -> Output(2, Softmax)
model_moons = TwoLayerNeuralNetwork(in_features=2, hidden_dim=16, out_classes=2, seed=42)

# Training Hyperparameters
epochs = 250
learning_rate = 0.5
loss_history = []
acc_history = []

# 3. The 5-Step Training Loop
print("=" * 65)
print("TRAINING 2-LAYER NEURAL NETWORK FROM SCRATCH ON MOONS")
print("=" * 65)

for epoch in range(epochs):
    # Step 1: Forward Pass
    probs = model_moons.forward(X_moons)
    
    # Step 2: Compute Loss & Accuracy
    loss = model_moons.compute_loss(probs, Y_moons_onehot)
    preds = np.argmax(probs, axis=1)
    acc = np.mean(preds == y_moons)
    loss_history.append(loss)
    acc_history.append(acc)
    
    # Step 3: Backward Pass
    grads = model_moons.backward(Y_moons_onehot)
    
    # Step 4: Gradient Descent Weight Updates
    model_moons.W1 -= learning_rate * grads['dW1']
    model_moons.b1 -= learning_rate * grads['db1']
    model_moons.W2 -= learning_rate * grads['dW2']
    model_moons.b2 -= learning_rate * grads['db2']
    
    if (epoch + 1) % 50 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:<4} | Loss: {loss:.4f} | Training Accuracy: {acc*100:.1f}%")

print("=" * 65)

# 4. Visualization: Loss Curve and Learned Non-Linear Decision Boundary
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Convergence Curves
axes[0].plot(range(1, epochs + 1), loss_history, color='#ef4444', lw=2.5, label='Cross-Entropy Loss')
axes[0].set_ylabel('Loss', color='#ef4444', fontsize=11)
ax_acc = axes[0].twinx()
ax_acc.plot(range(1, epochs + 1), [a * 100 for a in acc_history], color='#2563eb', lw=2.5, linestyle='--', label='Accuracy (%)')
ax_acc.set_ylabel('Accuracy (%)', color='#2563eb', fontsize=11)
axes[0].set_title('Training Convergence (Loss & Accuracy)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].grid(True, linestyle='--', alpha=0.4)

# Subplot 2: Non-Linear Decision Boundary Surface
x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid_data = np.c_[xx.ravel(), yy.ravel()]

grid_probs = model_moons.forward(grid_data)[:, 1].reshape(xx.shape)

contour = axes[1].contourf(xx, yy, grid_probs, levels=25, cmap='RdBu_r', alpha=0.7)
axes[1].contour(xx, yy, grid_probs, levels=[0.5], colors=['#0f172a'], linewidths=2.5)
scatter0 = axes[1].scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], color='#ef4444', edgecolors='k', s=45, label='Class 0')
scatter1 = axes[1].scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], color='#3b82f6', edgecolors='k', s=45, label='Class 1')
axes[1].set_title(f'Learned Decision Surface (Final Acc: {acc_history[-1]*100:.1f}%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Feature x1')
axes[1].set_ylabel('Feature x2')
axes[1].legend(loc='upper right')
plt.colorbar(contour, ax=axes[1], label='Probability P(Class 1)')

plt.tight_layout()
fig_path = os.path.join(assets_dir, 'backprop_moons_decision_boundary.png')
plt.savefig(fig_path, dpi=120)
plt.close()
print(f"Figure successfully saved to: {fig_path}")


TRAINING 2-LAYER NEURAL NETWORK FROM SCRATCH ON MOONS
Epoch 1    | Loss: 0.5507 | Training Accuracy: 77.0%
Epoch 50   | Loss: 0.2422 | Training Accuracy: 87.8%
Epoch 100  | Loss: 0.2083 | Training Accuracy: 91.5%
Epoch 150  | Loss: 0.1655 | Training Accuracy: 93.8%
Epoch 200  | Loss: 0.1218 | Training Accuracy: 95.8%
Epoch 250  | Loss: 0.0914 | Training Accuracy: 97.5%
Figure successfully saved to: Module-05-Neural-Networks/assets\backprop_moons_decision_boundary.png


---

## 📝 Chapter Summary

| Stage / Concept | Plain English Meaning | Mathematical Equation | Key Takeaway |
|:---|:---|:---|:---|
| **Forward Pass** | Computes predictions from inputs | $Z^{[l]} = A^{[l-1]} W^{[l]} + \mathbf{b}^{[l]}$ | Must cache intermediate tensors in memory for backprop. |
| **Output Error ($\delta^{[L]}$)** | Raw error driving the backward pass | $\delta^{[L]} = \frac{1}{N}(\hat{Y} - Y)$ | Softmax + CCE derivative cancels out denominators. |
| **Weight Gradient ($\nabla_W$)** | Upstream activations dot output error | $\frac{\partial J}{\partial W^{[l]}} = (A^{[l-1]})^T \delta^{[l]}$ | Matches weight matrix dimensions exactly. |
| **Bias Gradient ($\nabla_{\mathbf{b}}$)** | Accumulation of errors across batch | $\frac{\partial J}{\partial \mathbf{b}^{[l]}} = \sum_{i=1}^N \delta_i^{[l]}$ | Summed down batch dimension (axis 0). |
| **Hidden Error ($\delta^{[l-1]}$)** | Error pulled through weights & activation | $\delta^{[l-1]} = (\delta^{[l]} (W^{[l]})^T) \odot \sigma'$ | Propagates error deeper into early layers. |
| **Gradient Checking** | Two-sided finite difference verification | $\frac{J(\theta+\epsilon) - J(\theta-\epsilon)}{2\epsilon}$ | Relative error $< 10^{-7}$ proves zero bugs. |

---


## 🔗 Resources & Further Reading

| Resource | Type | Why Read It |
|:---|:---|:---|
| **[Learning Internal Representations by Error Propagation (Rumelhart, Hinton, & Williams, 1986)](https://www.nature.com/articles/323533a0)** | Seminal Paper | The landmark paper that introduced backpropagation to the world. |
| **[CS231n: Backpropagation and Vectorized Operations (Andrej Karpathy)](https://cs231n.github.io/optimization-2/)** | Landmark Guide | Karpathy's masterclass tutorial on intuitive circuit backpropagation and gate analysis. |
| **[Calculus on Computational Graphs (Chris Olah)](https://colah.github.io/posts/2015-08-Backprop/)** | Visual Article | Beautiful visual explanation of reverse-mode automatic differentiation. |
| **[Automatic Differentiation in Machine Learning: A Survey (Baydin et al., 2018)](https://arxiv.org/abs/1502.05767)** | Survey Paper | Comprehensive survey of forward vs reverse mode automatic differentiation. |

---


## ➡️ Next Chapter: Chapter 5.6 — Gradient Descent & Optimizers

Now that we have computed exact gradients $\nabla W$, how should we use them to update weights?
- **Batch vs SGD vs Mini-Batch:** The speed vs stability trade-off that defines modern machine learning.
- **Momentum:** Escaping shallow local minima and plowing through ravines like a heavy rolling bowling ball.
- **RMSprop & AdaGrad:** Adaptive per-parameter learning rates for sparse features.
- **Adam (Adaptive Moment Estimation):** Combining Momentum + RMSprop + Bias Correction — the undisputed champion optimizer of deep learning.
- **Learning Rate Schedules:** Step decay, Warmup, and Cosine Annealing.

---
*Made with ❤️ | Zero to Hero — AI Engineer Learning Path*
